# 2. The tool broker: mediating every tool call

**What you'll learn:** how to register a tool with AgentVanguard so every
call to it is checked automatically -- including the cases where the check
should *fail*: an unapproved high-risk action, an SSRF attempt, and a
path-traversal attempt.

## Prerequisites

Same as notebook 1: `pip install -e ".[notebooks]"`, no network/API keys
needed.

## The core idea: `ToolManifest` + `@firewall.tool`

A `ToolManifest` is a short declaration of what a tool does and how risky
it is:

| Field | Meaning |
|---|---|
| `name` | Must match how the tool will be looked up/called |
| `action_type` | `READ` / `WRITE` / `EXECUTE` / `SEND_EXTERNAL` / `DELETE` / `FINANCIAL` |
| `side_effect` | How far the blast radius goes, e.g. `EXTERNAL_COMMUNICATION` |
| `risk_tier` | `LOW` / `MEDIUM` / `HIGH` / `CRITICAL` -- drives the base risk score |
| `requires_approval` | If `True`, every call needs human sign-off (see 2.2 below) |
| `allowed_callers` | Which named agents may call this tool at all |

Decorating a plain Python function with `@firewall.tool(manifest=...)`
returns a wrapped version that, on every call, builds a `PolicyInput`,
authorizes it, and only runs your original function if the decision is
`ALLOW`/`ALLOW_WITH_MODIFICATIONS` -- otherwise it raises
`VanguardBlockedError`.

You'll see a `_context={...}` keyword on every call below. It's how the
caller tells AgentVanguard *who* is calling (`agent=...`) and *which
conversation* this belongs to (`session_id=...`); it's popped off before
your wrapped function ever sees it, so your function's own signature stays
unchanged.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from agentvanguard import ActionType, AgentVanguard, AgentIdentity, RiskTier, SideEffect, ToolManifest
from agentvanguard.core.errors import VanguardBlockedError

CONFIG_PATH = str(REPO_ROOT / "examples" / "agentvanguard.yaml")
firewall = AgentVanguard.from_config(CONFIG_PATH)

# `AgentIdentity` is who's making the call -- checked against each tool's
# `allowed_callers` and the workflow's `allowed_agents`/`allowed_tools`.
SUPPORT_AGENT = AgentIdentity(name="support_agent", workflow="customer_support")
CONTEXT = {"agent": SUPPORT_AGENT, "session_id": "notebook-tool-broker"}


## 2.1 A low-risk tool: auto-allowed

`lookup_order` is declared `risk_tier: low` here, with `support_agent` as
an allowed caller. A call that passes schema/content checks and comes from
an allowed caller is let straight through -- the wrapped function runs and
its return value is handed back unchanged (after also being scanned by the
*output* guard, which finds nothing sensitive in this particular result).

In [2]:
@firewall.tool(
    manifest=ToolManifest(
        name="lookup_order",
        action_type=ActionType.READ,
        side_effect=SideEffect.INTERNAL_READ,
        risk_tier=RiskTier.LOW,
        allowed_callers=["support_agent"],
    )
)
def lookup_order(order_id: str) -> str:
    return f"Order {order_id} shipped to alice@example.com, status: delivered."


print(lookup_order(order_id="1234", _context=CONTEXT))


Order 1234 shipped to alice@example.com, status: delivered.


## 2.2 A high-risk tool: requires approval

`send_email` is already declared under `tools:` in
`examples/agentvanguard.yaml`, with `risk_tier: high` and
`requires_approval: true`. Rather than re-declare its manifest, we look it
up by name with `firewall.wrap_tool("send_email", fn)`.

`requires_approval: true` means every call is escalated to
`REQUIRE_USER_CONFIRMATION`, which is then handed to the configured
**approval provider** to resolve. Our example config uses
`approval_provider: auto_deny` -- the safe, non-interactive default that
*never* grants approval (the alternative, `cli_prompt`, pauses and asks a
human at the terminal; see `docs/architecture.md` for other providers). So
in practice this call is always blocked, surfaced to us as a
`VanguardBlockedError`.

In [3]:
def _send_email(to: str, body: str) -> str:
    print(f"[would send email to {to}]: {body}")
    return "sent"


send_email = firewall.wrap_tool("send_email", _send_email)

try:
    send_email(to="alice@example.com", body="Your refund has been processed.", _context=CONTEXT)
except VanguardBlockedError as exc:
    print("blocked:", exc.decision.decision.value)
    print("reason: ", exc.decision.reasons[0])


blocked: REQUIRE_USER_CONFIRMATION
reason:  tool 'send_email' requires user approval (risk_tier=high)


## 2.3 SSRF: a tool argument pointing at the cloud metadata service

Imagine an agent that can check whether a vendor's status page is up. A
prompt-injected document could try to trick it into instead requesting
`169.254.169.254` -- the cloud-provider instance metadata address, which
serves credentials over plain HTTP with no auth on AWS/GCP/Azure-style
platforms. This is a real, common SSRF target.

The tool-call guard inspects every string argument that looks like a
URL/host, resolves it, and rejects private/loopback/link-local addresses
*before* your function body ever runs -- so `check_status_page` below
never actually makes the request.

In [4]:
@firewall.tool(
    manifest=ToolManifest(
        name="check_status_page",
        action_type=ActionType.READ,
        side_effect=SideEffect.EXTERNAL_COMMUNICATION,
        risk_tier=RiskTier.MEDIUM,
        allowed_callers=["support_agent"],
    )
)
def check_status_page(url: str) -> str:
    return f"checked {url}: 200 OK"  # never actually reached for a blocked URL


try:
    check_status_page(url="http://169.254.169.254/latest/meta-data/", _context=CONTEXT)
except VanguardBlockedError as exc:
    print("blocked:", exc.decision.decision.value)
    for reason in exc.decision.reasons:
        print("reason: ", reason)


blocked: DENY
reason:  argument 'url' targets a non-public address: 169.254.169.254
reason:  argument 'url' host '169.254.169.254' is not in the egress allowlist


## 2.4 Path traversal: a tool argument escaping the workspace

`read_file` is scoped to `tool_guard.workspace_root` in config (`.`, the
repo root, in our example). An argument like `../../etc/passwd` normalizes
to a path outside that root and is rejected -- regardless of what the
agent's own reasoning concluded, or how plausible the request sounded.

In [5]:
@firewall.tool(
    manifest=ToolManifest(
        name="read_file",
        action_type=ActionType.READ,
        side_effect=SideEffect.INTERNAL_READ,
        risk_tier=RiskTier.LOW,
        allowed_callers=["support_agent"],
    )
)
def read_file(path: str) -> str:
    return f"contents of {path}"  # never actually reached for a blocked path


try:
    read_file(path="../../etc/passwd", _context=CONTEXT)
except VanguardBlockedError as exc:
    print("blocked:", exc.decision.decision.value)
    for reason in exc.decision.reasons:
        print("reason: ", reason)


blocked: DENY
reason:  argument 'path' resolves outside the workspace root: '../../etc/passwd'


## What's next

- **`03_sessions_limits_audit.ipynb`** -- what happens when an agent keeps
  calling the same tool over and over, and how to inspect the resulting
  audit trail.
- **`04_full_agent_loop.ipynb`** -- combining `wrap_agent` with the tools
  registered here, in one end-to-end mock conversation.